# Bank Loan Default Risk Analysis
**Beginner-friendly, self-contained project**

This notebook explores loan data and trains a baseline machine-learning model to estimate whether a loan may default.

> **Important:** `loan_default_sample.csv` is a synthetic teaching dataset created for this project. It is not real customer or bank data. Results are for demonstration only and must not be used for actual lending decisions.


## 1. Import libraries
Run the notebook from top to bottom. If a package is missing, install it using the optional cell below.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, ConfusionMatrixDisplay
)

print("Libraries imported successfully.")


## 2. Load the dataset
The CSV is included in this repository. If you run this notebook in Google Colab, upload the CSV and keep it in the same working folder, or use the fallback below to create a sample dataset automatically.


In [ ]:
from pathlib import Path

data_path = Path("loan_default_sample.csv")
if data_path.exists():
    df = pd.read_csv(data_path)
else:
    # Fallback sample data so the notebook can still run if the CSV was not uploaded.
    rng = np.random.default_rng(42)
    n = 1200
    age = rng.integers(21, 66, n)
    income = np.clip(rng.normal(65000, 22000, n), 15000, 180000).round().astype(int)
    loan_amount = np.clip(rng.normal(18000, 9000, n), 1000, 60000).round().astype(int)
    credit_score = np.clip(rng.normal(680, 65, n), 350, 850).round().astype(int)
    employment_years = np.clip(rng.normal(7, 5, n), 0, 40).round().astype(int)
    dti = np.clip(rng.normal(.32, .14, n), .03, .85).round(3)
    interest = np.clip(17 - (credit_score - 350) / 50 + rng.normal(0, 1.5, n), 4, 30).round(2)
    previous = rng.binomial(1, .12, n)
    home = rng.choice(["Rent", "Own", "Mortgage"], n, p=[.38, .22, .40])
    purpose = rng.choice(["Personal", "Education", "Medical", "Home Improvement", "Debt Consolidation"], n)
    logit = -2 + 2.8*dti + .000018*loan_amount - .000018*income - .008*(credit_score-600) + 1.25*previous + .07*(interest-12) - .018*employment_years
    p = 1/(1+np.exp(-logit))
    default = rng.binomial(1, p)
    df = pd.DataFrame({
        "age": age, "annual_income": income, "loan_amount": loan_amount,
        "credit_score": credit_score, "employment_years": employment_years,
        "debt_to_income_ratio": dti, "interest_rate": interest,
        "previous_defaults": previous, "home_ownership": home,
        "loan_purpose": purpose, "default": default
    })

print("Dataset loaded:", df.shape)
display(df.head())


## 3. Understand and clean the data
`default` is the target column: 1 means default in this synthetic example, 0 means no default.


In [ ]:
print("Rows and columns:", df.shape)
print("\\nColumn types:")
display(df.dtypes.to_frame("data_type"))
print("\\nMissing values per column:")
display(df.isna().sum().to_frame("missing_values"))
print("\\nDuplicate rows:", df.duplicated().sum())
print("\\nTarget distribution:")
display(df["default"].value_counts().rename(index={0:"No default", 1:"Default"}).to_frame("count"))


## 4. Exploratory data analysis (EDA)


In [ ]:
df.describe(include="all").T


In [ ]:
df["default"].value_counts().sort_index().plot(kind="bar")
plt.title("Loan Default Class Distribution")
plt.xlabel("Default (0 = No, 1 = Yes)")
plt.ylabel("Number of records")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(df["credit_score"], bins=20, edgecolor="black")
plt.title("Credit Score Distribution")
plt.xlabel("Credit score")
plt.ylabel("Number of borrowers")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(7, 4))
for label, group in df.groupby("default"):
    plt.hist(group["debt_to_income_ratio"], bins=20, alpha=.55, label=f"Default = {label}")
plt.title("Debt-to-Income Ratio by Default Status")
plt.xlabel("Debt-to-income ratio")
plt.ylabel("Number of borrowers")
plt.legend()
plt.tight_layout()
plt.show()


## 5. Train a baseline model
We use a train/test split and a preprocessing pipeline so numeric and categorical columns are handled consistently. Logistic regression is a simple baseline, not a final banking-grade risk system.


In [ ]:
X = df.drop(columns=["default"])
y = df["default"]

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]
print("Model trained successfully.")


## 6. Evaluate the model
Accuracy alone can be misleading when defaults are uncommon, so we also report precision, recall, F1 and ROC-AUC.


In [ ]:
metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1 score": f1_score(y_test, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_prob) if y_test.nunique() == 2 else np.nan
}
metrics_df = pd.DataFrame({"Metric": list(metrics.keys()), "Score": list(metrics.values())})
display(metrics_df.round(3))
print(classification_report(y_test, y_pred, zero_division=0))


In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()


## 7. Example prediction
This is an illustrative prediction from the trained model. It is not financial advice and should not be used to approve or reject a real loan.


In [ ]:
example_borrower = pd.DataFrame([{
    "age": 35,
    "annual_income": 55000,
    "loan_amount": 20000,
    "credit_score": 640,
    "employment_years": 4,
    "debt_to_income_ratio": 0.42,
    "interest_rate": 14.5,
    "previous_defaults": 0,
    "home_ownership": "Rent",
    "loan_purpose": "Personal"
}])

default_probability = model.predict_proba(example_borrower)[0, 1]
print(f"Illustrative estimated default probability: {default_probability:.1%}")
print("Reminder: demonstration only; synthetic data and a baseline model.")


## 8. Conclusion
- The notebook demonstrates data loading, basic cleaning checks, EDA, visualizations and a baseline classification model.
- The metrics shown are specific to this synthetic dataset and random train/test split.
- A real-world project would require approved, representative data; leakage checks; fairness and bias analysis; calibration; validation over time; and review by qualified risk professionals.

## Project author
Gazi Abbas Rizvi
